# Week 19, Day 1: Merge sort

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w019d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import math
for n in (1_000, 1_000_000):
    print(n, n * n, round(n * math.log2(n)))

In [ ]:
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:          # <= takes from the left on ties: stable
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

def merge_sort(items):
    if len(items) <= 1:
        return items[:]
    mid = len(items) // 2
    return merge(merge_sort(items[:mid]), merge_sort(items[mid:]))

print(merge_sort([38, 27, 43, 3, 9, 82, 10]))

*Interactive figure on the website: Merge sort splits until pieces have one item, then merges pairs of sorted pieces into bigger sorted pieces. Compare the counts with last week's sorts on reversed data..*

In [ ]:
def merge_sort_counting(items):
    """Return (sorted list, comparisons made)."""
    if len(items) <= 1:
        return items[:], 0
    mid = len(items) // 2
    left, a = merge_sort_counting(items[:mid])
    right, b = merge_sort_counting(items[mid:])
    result, i, j, c = [], 0, 0, 0
    while i < len(left) and j < len(right):
        c += 1
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result += left[i:] + right[j:]
    return result, a + b + c

import math, random
for n in (16, 1024, 65536):
    data = random.Random(1).sample(range(n), n)
    print(n, merge_sort_counting(data)[1], round(n * math.log2(n)), n * (n - 1) // 2)

In [ ]:
def merge_sort(items):
    buffer = items[:]                         # one spare list for the whole sort
    def sort(lo, hi):                         # sort items[lo:hi] in place
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        sort(lo, mid)
        sort(mid, hi)
        buffer[lo:hi] = items[lo:hi]          # copy the two sorted halves out...
        i, j = lo, mid
        for k in range(lo, hi):               # ...and merge them back into items
            if j >= hi or (i < mid and buffer[i] <= buffer[j]):
                items[k] = buffer[i]
                i += 1
            else:
                items[k] = buffer[j]
                j += 1
    sort(0, len(items))

data = [38, 27, 43, 3, 9, 82, 10]
merge_sort(data)
print(data)

In [ ]:
def merge_runs(items, buffer, lo, mid, hi):
    buffer[lo:hi] = items[lo:hi]
    i, j = lo, mid
    for k in range(lo, hi):
        if j >= hi or (i < mid and buffer[i] <= buffer[j]):
            items[k] = buffer[i]; i += 1
        else:
            items[k] = buffer[j]; j += 1

def merge_sort_bottom_up(items):
    n = len(items)
    buffer = items[:]
    width = 1
    while width < n:
        for lo in range(0, n, 2 * width):
            mid = min(lo + width, n)
            hi = min(lo + 2 * width, n)
            if mid < hi:
                merge_runs(items, buffer, lo, mid, hi)
        width *= 2

data = [38, 27, 43, 3, 9, 82, 10]
merge_sort_bottom_up(data)
print(data)

In [ ]:
import heapq

march = [3, 9, 14]
april = [1, 10, 22]
may = [5, 6, 30]
print(list(heapq.merge(march, april, may)))

In [ ]:
import math
for n in (10, 1000, 1_000_000):
    print(n, round(math.log2(math.factorial(n)) if n <= 1000 else n * math.log2(n) - 1.4427 * n), round(n * math.log2(n)))

> **Key idea.** Merge sort: split in half, sort each half, merge. About n log₂ n comparisons in every case, stable, O(n) extra space. No comparison sort can do better than about n log₂ n in the worst case.

### Exercise: Merge sort by slicing

Write `merge(left, right)` and `merge_sort(items)` (returning a new sorted list, the original unchanged, stable). Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for data in ([38, 27, 43, 3, 9, 82, 10], [], [1], [2, 1], [3, 3, 1], [\"kiwi\", \"fig\", \"apple\"]):\n        before = list(data)\n        assert merge_sort(data) == sorted(data) and data == before\n    rng = random.Random(4)\n    data = [rng.randint(-1000, 1000) for _ in range(20000)]\n    assert merge_sort(data) == sorted(data)\n    assert merge([1, 4], [2, 3, 5]) == [1, 2, 3, 4, 5]\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Exercise: Count the comparisons

Write `comparisons(items)` that merge sorts a copy and returns how many comparisons the merges made. Then set `sorted_1024` to the count for `list(range(1024))` and `reversed_1024` for `list(range(1024, 0, -1))` (run your function), and `levels_1024` to how many levels of splitting there are for 1,024 items.

In [ ]:
sorted_1024 = ___
reversed_1024 = ___
levels_1024 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert comparisons([2, 1]) == 1 and comparisons([]) == 0 and comparisons([1, 2, 3, 4]) == 4\n    assert sorted_1024 == comparisons(list(range(1024))) == 5120\n    assert reversed_1024 == comparisons(list(range(1024, 0, -1))) == 5120\n    assert levels_1024 == 10")

### Exercise: One buffer

Write `merge_sort_in_place(items, key=lambda x: x)` that sorts the list it's given (and returns nothing), using a recursive helper `sort(lo, hi)` on positions and **one** buffer list made once at the start: no slicing inside the recursion except copying into the buffer. It must be stable by `key`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for data in ([38, 27, 43, 3, 9, 82, 10], [], [1], [2, 1], [5, 5, 5]):\n        copy = list(data)\n        assert merge_sort_in_place(copy) is None and copy == sorted(data)\n    rng = random.Random(2)\n    recs = [(rng.randint(0, 9), i) for i in range(3000)]\n    copy = recs[:]\n    merge_sort_in_place(copy, key=lambda r: r[0])\n    assert copy == sorted(recs, key=lambda r: r[0])\n    big = [rng.random() for _ in range(50_000)]\n    copy = big[:]\n    merge_sort_in_place(copy)\n    assert copy == sorted(big)")

### Exercise: Bottom up

Write `merge_sort_bottom_up(items)` that sorts the list in place without recursion: merge runs of width 1, 2, 4, ... until the width reaches the length. Then set `passes_1000` to how many times the width loop runs for 1,000 items.

In [ ]:
passes_1000 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for n in (0, 1, 2, 3, 7, 8, 9, 100, 1000):\n        data = random.Random(n).sample(range(10 * n + 1), n)\n        copy = data[:]\n        merge_sort_bottom_up(copy)\n        assert copy == sorted(data), n\n    assert passes_1000 == 10\n    assert \"merge_sort_bottom_up(\" not in source.split(\"def merge_sort_bottom_up\", 1)[1]")

### Exercise: Merge many

Write `merge_all(lists)` that merges any number of sorted lists into one sorted list, by merging them in **pairs**, round after round (the first with the second, the third with the fourth, ...), until one is left. Don't use `sort`, `sorted` or `heapq`. With k lists of n items in total, this is O(n log k).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert merge_all([[3, 9, 14], [1, 10, 22], [5, 6, 30]]) == [1, 3, 5, 6, 9, 10, 14, 22, 30]\n    assert merge_all([]) == [] and merge_all([[], [2], []]) == [2]\n    rng = random.Random(6)\n    lists = []\n    for _ in range(37):\n        xs = [rng.randint(0, 999) for _ in range(rng.randint(0, 50))]\n        xs.sort()\n        lists.append(xs)\n    assert merge_all(lists) == sorted(x for xs in lists for x in xs)\n    import re\n    body = source\n    assert \"heapq\" not in body and not re.search(r\"\\bsorted\\(|\\.sort\\(\", body)")

### Bug hunt: Not stable

This merge sort sorts correctly, but swaps the order of equal keys. It should print `[(1, 'b'), (2, 'a'), (2, 'c')]`. Fix the one character that makes it unstable.

In [ ]:
def merge_sort(items, key):
    if len(items) <= 1:
        return items[:]
    mid = len(items) // 2
    left, right = merge_sort(items[:mid], key), merge_sort(items[mid:], key)
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if key(left[i]) < key(right[j]):
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

print(merge_sort([(2, "a"), (1, "b"), (2, "c")], key=lambda p: p[0]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stable\"\n    expect_output(\"[(1, 'b'), (2, 'a'), (2, 'c')]\")")

### Bug hunt: Lost the leftovers

This merge loses items: `merge([1, 5, 9], [2, 3])` gives `[1, 2, 3]`. Fix it so it prints `[1, 2, 3, 5, 9]`.

In [ ]:
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    return result

print(merge([1, 5, 9], [2, 3]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"all items\"\n    expect_output(\"[1, 2, 3, 5, 9]\")")

### Puzzle: Merge sort

Arrange the lines for `merge_sort`, given a working `merge`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # comparisons made by merge([1, 2, 3], [4, 5, 6])
b = ___   # comparisons made by merge([1, 3, 5], [2, 4, 6])
c = ___   # levels of splitting for 64 items
d = ___   # True or False: merge sort is faster on already sorted data than on random data (in Big-O terms)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 3 and b == 5 and c == 6 and d is False")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: sorting a file in chunks

`numbers.txt` holds one whole number per line. Pretend memory only holds `chunk` numbers at a time, and sort it the way databases sort huge tables. Write `external_sort(source, target, chunk)`:

1. Read the source file `chunk` lines at a time; sort each chunk (you may use `sorted` here) and write it to its own file `run0.txt`, `run1.txt`, ... (one number per line).
2. Merge all the run files into `target`, one number per line, by reading **one line at a time from each run** (keep a list of open files and their current front numbers; repeatedly write the smallest front number and read the next line from that run).
3. Return the number of runs.

Don't read a whole run or the whole source into memory at once (the checks look for `.read()` and `.readlines()`).

In [ ]:
# Run this cell first: it creates the files used below.
with open('numbers.txt', 'w', encoding='utf-8') as f:
    f.write('42\n7\n19\n3\n88\n25\n11\n64\n5\n30\n')

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_small():\n    \"small\"\n    runs = external_sort(\"numbers.txt\", \"sorted.txt\", 3)\n    assert runs == 4\n    assert read_file(\"sorted.txt\").split() == [str(x) for x in sorted([42, 7, 19, 3, 88, 25, 11, 64, 5, 30])]\n    assert read_file(\"run0.txt\").split() == [\"7\", \"19\", \"42\"]\ndef test_big():\n    \"bigger\"\n    rng = random.Random(3)\n    nums = [rng.randint(-10_000, 10_000) for _ in range(5000)]\n    write_file(\"big.txt\", \"\\n\".join(map(str, nums)) + \"\\n\")\n    assert external_sort(\"big.txt\", \"big_sorted.txt\", 700) == 8\n    assert [int(x) for x in read_file(\"big_sorted.txt\").split()] == sorted(nums)\ndef test_rules():\n    \"streams\"\n    assert \".read()\" not in source and \"readlines\" not in source")

## Warm-up

For a million items, n² is a million million, and n log₂ n is about twenty million: 50,000 times less. That gap is the difference between an hour and a fraction of a second, and it's what merge sort buys.

## Merge sort again

The three steps, as you saw in Week 16:

1. **Divide:** split the list into two halves.
2. **Conquer:** sort each half, recursively. A list of 0 or 1 items is already sorted.
3. **Combine:** merge the two sorted halves, by repeatedly taking the smaller of the two front items.

Step through it. The faded bars are outside the part being worked on:

## Why n log n

Picture every call as a node in a tree. The top call works on all n items; it makes two calls on n/2 each; those make four calls on n/4; and so on, down to n calls on single items.

```
level 0:            [ n ]                       1 merge of n items
level 1:       [n/2]     [n/2]                  2 merges of n/2
level 2:    [n/4] [n/4] [n/4] [n/4]             4 merges of n/4
  ...
level k:   n pieces of 1 item                   nothing to merge
```

- **Work per level:** merging pieces that add up to n items costs at most n comparisons, whatever the sizes of the pieces. Every level does about n work.
- **Number of levels:** the pieces halve each level, so there are about log₂ n levels before they reach size 1.

Total: about **n log₂ n** comparisons. And unlike insertion sort, that doesn't depend on the data: sorted, reversed or random, merge sort always splits the same way and always merges every level. It's **O(n log n) in the best, worst and average case**.

The comparisons stay just under n log₂ n, against n(n − 1)/2 for selection sort.

Merge sort's other properties:

- **Stable**, as long as the merge takes from the left on ties (`<=`).
- **Not in place:** the merge needs somewhere to put its result, **O(n) extra space**. That's its main cost.
- **Predictable:** no bad inputs, which is why it's used where a guaranteed time matters.

## Without the copies

The version above slices: `items[:mid]` and `items[mid:]` make new lists at every level, and `merge` builds another. That's correct and clear, but it allocates far more memory than needed. A faster version sorts **positions** `lo` to `hi` of one list, and merges through **one** spare list allocated once:

The merge condition reads: take from the left half if the right half is used up, or if the left still has items and its front item is not bigger. It now sorts **in place** from the caller's point of view (the list passed in ends up sorted), though it still uses O(n) extra space inside.

## Bottom up

Merge sort doesn't need recursion at all. Treat the list as n sorted runs of length 1. Merge neighbouring pairs into runs of 2, then runs of 2 into runs of 4, and so on, until one run covers everything:

The `while` loop runs about log₂ n times (width doubles), and each pass merges across the whole list: n log n again, with no recursion and no risk of hitting the recursion limit. The `min(..., n)` calls handle a last run that's shorter than the others.

## Merging many lists, and the speed limit

**Merging several sorted lists.** The merge idea extends to any number of sorted lists: repeatedly take the smallest of the front items. Python's `heapq.merge` does this efficiently (Week 47 shows how):

This is how computers sort data **too big for memory** (external sorting): read a chunk that fits, sort it, write it out as a sorted file; repeat for every chunk; then merge all the sorted files, reading each a little at a time. Databases sort huge tables this way. Today's challenge builds a small version with real files.

**The speed limit.** Can any sort beat n log n? Not one that only learns about the data by **comparing** two items at a time. Here's the argument:

- A list of n different items could be in any of **n!** orders, and the sort must end up doing something different for each one.
- Each comparison has two outcomes, so it can at best halve the number of orders still possible.
- To get from n! possibilities down to 1 takes at least log₂(n!) comparisons, and log₂(n!) is about **n log₂ n − 1.44n**.

So merge sort is already within a whisker of the best possible comparison sort. The only way to go faster is to not compare at all, using what the values **are**: that's counting sort, on Day 3.

## Practice

## Quiz

## Daily challenge

## Before you go

Sort a shuffled deck of cards with merge sort: split into piles until each pile is one card, then merge pairs of piles, keeping each merged pile in order. Count the levels. Tomorrow: quicksort, which also splits the list in two, but does its hard work before the recursion instead of after.